# Análisis Espacial y Feature Engineering
En este notebook construiremos la grilla de 250m x 250m y realizaremos los cruces espaciales de nuestras capas limpias.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import os

PROC_DIR = '../01_DATOS/02_PROCESADOS'
MODEL_DIR = '../03_MODELO/01_dataset_modelo'

In [2]:
# 1. Cargar capas limpias
puntos = gpd.read_file(f"{PROC_DIR}/puntos_limpios.gpkg")
cestas = gpd.read_file(f"{PROC_DIR}/cestas_limpias.gpkg")
macrorutas = gpd.read_file(f"{PROC_DIR}/macrorutas_limpias.gpkg")
reportes = gpd.read_file(f"{PROC_DIR}/reportes_limpios.gpkg")

print("Datos cargados exitosamente.")

Datos cargados exitosamente.


In [3]:
# 2. Construir la Grilla Espacial de 250m x 250m
# Obtenemos el bounding box máximo que cubra los reportes y puntos críticos
minx, miny, maxx, maxy = reportes.total_bounds

# Ajustar bounds por los puntos críticos si están más lejos
minx2, miny2, maxx2, maxy2 = puntos.total_bounds
minx = min(minx, minx2)
miny = min(miny, miny2)
maxx = max(maxx, maxx2)
maxy = max(maxy, maxy2)

# Tamaño de celda en metros
cell_size = 250

grid_cells = []
for x0 in np.arange(minx, maxx, cell_size):
    for y0 in np.arange(miny, maxy, cell_size):
        x1 = x0 + cell_size
        y1 = y0 + cell_size
        grid_cells.append(box(x0, y0, x1, y1))

grilla = gpd.GeoDataFrame(grid_cells, columns=['geometry'], crs='EPSG:3116')
grilla['id_celda'] = [f"C{str(i).zfill(5)}" for i in range(len(grilla))]
print(f"Se generó una grilla de {len(grilla)} celdas de {cell_size}m x {cell_size}m.")

Se generó una grilla de 11094 celdas de 250m x 250m.


In [4]:
# 3. Cruce Espacial (Spatial Join)
# 3.1 Reportes por celda
reportes_grilla = gpd.sjoin(reportes, grilla, how='inner', predicate='intersects')
reportes_count = reportes_grilla.groupby('id_celda').size().reset_index(name='num_reportes')

# 3.2 Puntos Críticos (cercanos)
puntos_grilla = gpd.sjoin(puntos, grilla, how='inner', predicate='intersects')
puntos_count = puntos_grilla.groupby('id_celda').size().reset_index(name='num_puntos_criticos')

# 3.3 Cestas
cestas_grilla = gpd.sjoin(cestas, grilla, how='inner', predicate='intersects')
cestas_count = cestas_grilla.groupby('id_celda').size().reset_index(name='num_cestas')

# 3.4 Cobertura de barrido (Macrorutas)
macrorutas_grilla = gpd.sjoin(macrorutas, grilla, how='inner', predicate='intersects')
macrorutas_celdas = macrorutas_grilla['id_celda'].unique()


In [5]:
# 4. Consolidar variables en la grilla
grilla_modelo = grilla.merge(reportes_count, on='id_celda', how='left')
grilla_modelo = grilla_modelo.merge(puntos_count, on='id_celda', how='left')
grilla_modelo = grilla_modelo.merge(cestas_count, on='id_celda', how='left')

# Rellenar nulos con 0
grilla_modelo['num_reportes'] = grilla_modelo['num_reportes'].fillna(0)
grilla_modelo['num_puntos_criticos'] = grilla_modelo['num_puntos_criticos'].fillna(0)
grilla_modelo['num_cestas'] = grilla_modelo['num_cestas'].fillna(0)

# Variable de cobertura (1 = Sí, 0 = No)
grilla_modelo['tiene_macroruta'] = grilla_modelo['id_celda'].apply(lambda x: 1 if x in macrorutas_celdas else 0)

# Filtrar grilla para conservar solo zonas relevantes (ej. que tengan al menos 1 reporte, cesta, macroruta o punto)
grilla_activa = grilla_modelo[(grilla_modelo['num_reportes'] > 0) | 
                              (grilla_modelo['num_puntos_criticos'] > 0) | 
                              (grilla_modelo['tiene_macroruta'] == 1) | 
                              (grilla_modelo['num_cestas'] > 0)].copy()
print(f"Celdas activas en Bogotá (con algún dato de interés): {len(grilla_activa)}")

Celdas activas en Bogotá (con algún dato de interés): 5781


In [6]:
# 5. Distancias a infraestructuras (Distancia al elemento más cercano)
# Nota: Calcular distancias puede ser intensivo. 
# Usamos sjoin_nearest para encontrar la cesta y punto crítico más cercano al centroide de la celda.

centroides = grilla_activa.copy()
centroides.geometry = centroides.geometry.centroid

# Distancia a la cesta más cercana
if len(cestas) > 0:
    # El sjoin_nearest añade la distancia si especificamos distance_col
    dist_cestas = gpd.sjoin_nearest(centroides, cestas, how='left', distance_col='dist_cesta_mas_cercana')
    # Como puede devolver duplicados (empates de distancia), agrupamos y tomamos el mínimo real:
    min_dist_cestas = dist_cestas.groupby('id_celda')['dist_cesta_mas_cercana'].min().reset_index()
    grilla_activa = grilla_activa.merge(min_dist_cestas, on='id_celda', how='left')
else:
    grilla_activa['dist_cesta_mas_cercana'] = 9999

# Distancia a punto crítico más cercano
if len(puntos) > 0:
    dist_puntos = gpd.sjoin_nearest(centroides, puntos, how='left', distance_col='dist_punto_critico')
    min_dist_puntos = dist_puntos.groupby('id_celda')['dist_punto_critico'].min().reset_index()
    grilla_activa = grilla_activa.merge(min_dist_puntos, on='id_celda', how='left')
else:
    grilla_activa['dist_punto_critico'] = 9999

# Rellenar con un valor alto si no hay infra cercana en absoluto
grilla_activa['dist_cesta_mas_cercana'] = grilla_activa['dist_cesta_mas_cercana'].fillna(9999)
grilla_activa['dist_punto_critico'] = grilla_activa['dist_punto_critico'].fillna(9999)


In [7]:
# 6. Definición del Target y guardado del Dataset Analítico
# Definiremos un riesgo histórico simple por ahora para entrenar (ej. target = 1 si hay puntos críticos > 0 o reportes > umbral)
umbral_reportes = grilla_activa['num_reportes'].quantile(0.85) if grilla_activa['num_reportes'].max() > 0 else 0

grilla_activa['target_riesgo'] = np.where(
    (grilla_activa['num_puntos_criticos'] > 0) | (grilla_activa['num_reportes'] >= umbral_reportes),
    1, 0
)

print(f"Distribución del Target:\n{grilla_activa['target_riesgo'].value_counts()}")

# Guardar el dataset completo espacial
grilla_activa.to_file(f"{MODEL_DIR}/dataset_modelo.gpkg", driver='GPKG')

# Guardar versión CSV para Pandas directo (sin geometría)
grilla_df = grilla_activa.drop(columns='geometry')
grilla_df.to_csv(f"{MODEL_DIR}/dataset_modelo.csv", index=False)

print("Dataset Analítico (Grilla) creado y guardado en 03_MODELO/01_dataset_modelo/")

Distribución del Target:
target_riesgo
1    5781
Name: count, dtype: int64


Dataset Analítico (Grilla) creado y guardado en 03_MODELO/01_dataset_modelo/
